# Understanding Model Roles

This tutorial explains the three roles a model can play in ALF: **Oracle**, **Surrogate**, and **Generator**. Each role has different requirements and use cases.

## Setup

In [ ]:
import numpy as np
from alf_core.model.base_model import BaseModel
from alf_core.dataclasses import Candidate, LabelledCandidates, Predictions, Modality, TaskState
from alf_core.oracle.oracle import Oracle
from alf_core.surrogate.surrogate import Surrogate
from alf_core.optimizer.search import GeneratorSearch

## Example Model

Let's create a simple model that can fulfill all three roles:

In [ ]:
class SimpleModel(BaseModel):
    """A simple model that implements all required methods."""
    
    def __init__(self):
        self.coefficients = None
    
    def featurise(self, inputs):
        return np.array([c.data for c in inputs]).reshape(-1, 1)
    
    def train(self, train_data, val_data):
        X = self.featurise(train_data.candidates)
        y = train_data.labels
        self.coefficients = np.polyfit(X.flatten(), y, 2)
    
    def predict(self, candidate_points):
        X = self.featurise(candidate_points)
        means = np.polyval(self.coefficients, X.flatten())
        variances = np.random.rand(len(means)) * 0.1  # Mock variance
        return Predictions(means=means, variances=variances)
    
    def sample(self, condition=None):
        samples = np.random.uniform(-10, 10, size=10)
        return [Candidate(data=x, modality=Modality.TABULAR) for x in samples]

## Role 1: Oracle

**Purpose**: Evaluate candidates online during the active learning loop.

**When to use**: When you need real-time scoring of candidates (e.g., using a trained model as ground truth).

**Required methods**: `predict()`

In [ ]:
# Train a model to use as oracle
model = SimpleModel()
train_candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in [1.0, 2.0, 3.0]]
train_labels = np.array([1.0, 4.0, 9.0])
train_data = LabelledCandidates(candidates=train_candidates, labels=train_labels)
model.train(train_data, train_data)

# Wrap model as Oracle
oracle = Oracle(scorer=model)

# Evaluate new candidates
new_candidates = [Candidate(data=4.0, modality=Modality.TABULAR)]
state = TaskState(dataset=train_data, surrogate=None, round=0, acq_batch_size=1)
evaluated, updated_state = oracle.evaluate(new_candidates, state)

print(f"Oracle evaluated candidate: {evaluated.candidates[0].data} -> {evaluated.labels[0]:.2f}")

## Role 2: Surrogate

**Purpose**: Approximate expensive evaluation functions by learning from acquired data.

**When to use**: When direct evaluation is costly and you need a fast approximation for optimization.

**Required methods**: `train()`, `predict()`

**Optional methods**: `get_training_summary_metrics()`

In [ ]:
# Create surrogate model
surrogate_model = SimpleModel()
surrogate = Surrogate(model=surrogate_model)

# Fit surrogate on acquired data
train_candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in [1.0, 2.0, 3.0, 4.0]]
train_labels = np.array([1.0, 4.0, 9.0, 16.0])
train_data = LabelledCandidates(candidates=train_candidates, labels=train_labels)

val_candidates = [Candidate(data=2.5, modality=Modality.TABULAR)]
val_labels = np.array([6.25])
val_data = LabelledCandidates(candidates=val_candidates, labels=val_labels)

surrogate.fit(train_data, val_data)

# Make predictions with surrogate
test_candidates = [Candidate(data=5.0, modality=Modality.TABULAR)]
predictions = surrogate.predict(test_candidates)

print(f"Surrogate prediction: {predictions.means[0]:.2f} ± {np.sqrt(predictions.variances[0]):.2f}")
print(f"Training metrics: {surrogate.get_training_summary_metrics()}")

## Role 3: Generator

**Purpose**: Generate new candidate points to explore in the search space.

**When to use**: When the search space is continuous or you want a generative model to propose novel candidates.

**Required methods**: `sample()`

In [ ]:
# Use model as generator
generator_model = SimpleModel()
generator_search = GeneratorSearch(model=generator_model)

# Generate candidates
state = TaskState(dataset=train_data, surrogate=None, round=0, acq_batch_size=10)
generated_candidates = generator_search(state)

print(f"Generated {len(generated_candidates)} candidates")
print(f"Sample generated values: {[c.data for c in generated_candidates[:5]]}")

## Comparison Table

| Role | Required Methods | Optional Methods | Use Case | Example |
|------|------------------|------------------|----------|----------|
| **Oracle** | `predict()` | - | Online evaluation of candidates | Using a trained model to score generated candidates |
| **Surrogate** | `train()`, `predict()` | `get_training_summary_metrics()` | Approximate expensive functions | Training a neural network to approximate expensive simulations |
| **Generator** | `sample()` | - | Generate new candidates | Using a VAE/GAN to propose new molecular structures |

## Key Points

- **Same model, different roles**: A single `BaseModel` implementation can serve all three roles
- **Role determines wrapper**: Use `Oracle()`, `Surrogate()`, or `GeneratorSearch()` to wrap your model
- **Method requirements**: Only implement methods needed for your use case
- **Common pattern**: Surrogate approximates expensive Oracle evaluations
- **Flexibility**: Switch roles by changing the wrapper, not the model code

## Typical Active Learning Setup

1. **Oracle**: Expensive ground truth (could be a dataset or expensive model)
2. **Surrogate**: Fast approximation trained on acquired data from Oracle
3. **Search**: Generates candidate pool (could use Generator model or other search)
4. **Acquisition**: Scores candidates using Surrogate predictions
5. **Repeat**: Evaluate top candidates with Oracle, update Surrogate

See the experiment tutorials for complete examples:
- [Offline Design Tutorial](../experiments/offline_design_tutorial.ipynb)
- [Online Design Tutorial](../experiments/online_design_tutorial.ipynb)